# sonarwise: Audio Perception Engine
### Hear -  Search - Retrieve

**sonarwise** indexes any audio and makes it searchable by text, speaker, or sound events.

GitHub: [github.com/VK-Ant/sonarwise](https://github.com/VK-Ant/sonarwise) | PyPI: [pypi.org/project/sonarwise](https://pypi.org/project/sonarwise)

![sonarwise](https://raw.githubusercontent.com/VK-Ant/sonarwise/main/assets/sonarwise_ai.png)

## 1. Install

In [1]:
!pip install sonarwise openai-whisper transformers torch torchaudio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 3.8 MB/s eta 0:00:00


## 2. Upload your audio file
Upload any audio file: wav, mp3, m4a, flac, ogg

In [2]:
from google.colab import files
uploaded = files.upload()
audio_file = list(uploaded.keys())[0]
print(f"Uploaded: {audio_file}")

Saving data.m4a to data.m4a
Uploaded: data.m4a


## 3. Initialize sonarwise

In [3]:
import logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)

from sonarwise import SonarWise

sw = SonarWise(verbose=True)
print("sonarwise initialized!")

sonarwise initialized!


## 4. Index your audio

In [4]:
count = sw.index(audio_file)
print(f"\nIndexed {count} segments from {audio_file}")

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip


Processing:   0%|          | 0/1 [00:00<?, ?it/s]
  0%|                                               | 0.00/139M [00:00<?, ?iB/s]
  0%|▏                                      | 512k/139M [00:00<00:27, 5.24MiB/s]
  6%|██▍                                   | 8.98M/139M [00:00<00:02, 54.4MiB/s]
 18%|███████▏                               | 25.4M/139M [00:00<00:01, 108MiB/s]
 30%|███████████▋                           | 41.5M/139M [00:00<00:00, 132MiB/s]
 42%|████████████████▎                      | 58.1M/139M [00:00<00:00, 147MiB/s]
 54%|████████████████████▉                  | 74.3M/139M [00:00<00:00, 155MiB/s]
 65%|█████████████████████████▏             | 89.6M/139M [00:00<00:00, 156MiB/s]
 77%|██████████████████████████████▌         | 106M/139M [00:00<00:00, 161MiB/s]
 88%|███████████████████████████████████     | 121M/139M [00:00<00:00, 156MiB/s]
100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 139MiB/s]


preprocessor_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/555 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  776MB            

model.safetensors: downloading bytes:           |  0.00B            

Processing: 100%|██████████| 1/1 [00:44<00:00, 44.00s/it]


Indexed 1 segments from data.m4a


## 5. Search your audio by text

In [5]:
from sonarwise.utils.time_utils import ms_to_short

query = "Thanks"  # Change this to search for anything in your audio

results = sw.query(query, top_k=5)
print(f"Query: \"{query}\"\n")
for i, r in enumerate(results, 1):
    ts = f"{ms_to_short(r.start_ms)}-{ms_to_short(r.end_ms)}"
    print(f"{i}. [{ts}] {r.transcript} (score: {r.score})")

Query: "Thanks"

1. [0:00-0:05] Hi everyone, I am Makat Kumar. Thanks for giving this opportunity. (score: 0.2099)


## 6. View all indexed segments

In [6]:
import os
segments = sw._store.get_segments_by_file(os.path.abspath(audio_file))
print(f"Total segments: {len(segments)}\n")
for seg in segments:
    ts = f"{ms_to_short(seg.start_ms)}-{ms_to_short(seg.end_ms)}"
    print(f"[{ts}] {seg.transcript}")

Total segments: 1

[0:00-0:05] Hi everyone, I am Makat Kumar. Thanks for giving this opportunity.


## 7. Export to SRT subtitles

In [7]:
srt_path = sw.export(audio_file, format="srt", output="output.srt")
with open(srt_path) as f:
    print(f.read())

1
00:00:00,538 --> 00:00:05,482
[Unknown] Hi everyone, I am Makat Kumar. Thanks for giving this opportunity.




## 8. Export to JSON

In [8]:
import json
json_path = sw.export(audio_file, format="json", output="output.json")
with open(json_path) as f:
    data = json.load(f)
print(json.dumps(data[0], indent=2))  # Show first segment

{
  "segment_id": "88f46520-bbd",
  "filepath": "/content/data.m4a",
  "start_ms": 538,
  "end_ms": 5482,
  "duration_ms": 4944,
  "transcript": "Hi everyone, I am Makat Kumar. Thanks for giving this opportunity.",
  "speaker_id": null,
  "speaker_name": null,
  "event_tags": [],
  "language": "en",
  "indexed_at": "2026-07-29T12:39:01.120939"
}


## 9. Stats

In [9]:
stats = sw.stats()
print(f"Files:     {stats['total_files']}")
print(f"Segments:  {stats['total_segments']}")
print(f"Duration:  {ms_to_short(stats['total_duration_ms'])}")
print(f"Languages: {', '.join(stats['unique_languages'])}")
print(f"DB size:   {stats['db_size_mb']} MB")

Files:     1
Segments:  1
Duration:  0:04
Languages: en
DB size:   0.0 MB


## 10. Custom pluggable component example
Swap any model without changing the pipeline:

In [10]:
from sonarwise.core.transcriber import BaseTranscriber
from sonarwise.core.models import TranscriptResult

class MyCustomTranscriber(BaseTranscriber):
    """Replace with any ASR: Canary, Deepgram, Azure, your fine-tuned model."""
    def transcribe(self, audio):
        return TranscriptResult(text="custom output", language="en", confidence=1.0)

# Plug it in — same API, different model
sw_custom = SonarWise(transcriber=MyCustomTranscriber())
print("Custom transcriber plugged in!")
print("Every component is swappable: transcriber, embedder, store, diarizer, event classifier.")

Custom transcriber plugged in!
Every component is swappable: transcriber, embedder, store, diarizer, event classifier.


## Download results

In [11]:
files.download("output.srt")
files.download("output.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### sonarwise ecosystem

| Library | Purpose | Tagline |
|---------|---------|---------|  
| [SightRAG](https://github.com/VK-Ant/SightRAG) | Visual perception | See. Search. Retrieve. |
| **sonarwise** | Audio perception | Hear. Search. Retrieve. |
| [adaptive-intelligence](https://github.com/VK-Ant/adaptive-intelligence) | Reasoning & memory | Learn. Remember. Adapt. |
| [llmevalkit](https://github.com/VK-Ant/llmevalkit) | Evaluation | Evaluate. Score. Improve. |

**GitHub:** [github.com/VK-Ant/sonarwise](https://github.com/VK-Ant/sonarwise) | **PyPI:** `pip install sonarwise`